# Require Libraries And Modules

In [40]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind
from sklearn.preprocessing import StandardScaler
from scipy import stats

In [4]:
df=pd.read_csv("NFLX.csv")
df.head()

,Date,Open,High,Low,Close,Adj Close,Volume
0,2018-02-05,262.000000,267.899994,250.029999,254.259995,254.259995,11896100
1,2018-02-06,247.699997,266.700012,245.000000,265.720001,265.720001,12595800
2,2018-02-07,266.579987,272.450012,264.329987,264.559998,264.559998,8981500
3,2018-02-08,267.079987,267.619995,250.000000,250.100006,250.100006,9306700
4,2018-02-09,253.850006,255.800003,236.110001,249.470001,249.470001,16906900


In [5]:
df.dtypes

Date          object
Open         float64
High         float64
Low          float64
Close        float64
Adj Close    float64
Volume         int64
dtype: object

# Convert Date to Datetime

In [6]:
# Convert the 'Date' column to datetime format
df['Date'] = pd.to_datetime(df['Date'])

# Check the updated data types
print(df.dtypes)


Date         datetime64[ns]
Open                float64
High                float64
Low                 float64
Close               float64
Adj Close           float64
Volume                int64
dtype: object


# Handle Missing Values

In [8]:
df.isnull().sum()

Date         0
Open         0
High         0
Low          0
Close        0
Adj Close    0
Volume       0
dtype: int64

# Address Outliers


In [13]:
Q1 = df['Volume'].quantile(0.25)
Q3 = df['Volume'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers = df[(df['Volume'] < lower_bound) | (df['Volume'] > upper_bound)]
print(f"Number of outliers: {outliers.shape[0]}")


Number of outliers: 57


# Handle Outliers by Log Transformation:

In [15]:
df['Volume'] = np.log1p(df['Volume'])
df['Volume'] 

0       16.291721
1       16.348874
2       16.010678
3       16.046245
4       16.643232
          ...    
1004    16.813615
1005    16.930904
1006    16.478982
1007    16.108571
1008    15.867375
Name: Volume, Length: 1009, dtype: float64

In [26]:
from scipy.stats import ttest_ind

# Define a function to detect outliers using IQR
def detect_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

# List of numerical columns to check for outliers
numerical_columns = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']

# Loop through all numerical columns
for col in numerical_columns:
    # Detect outliers for each column
    outliers, lower_bound, upper_bound = detect_outliers(df, col)
    
    # Get non-outlier values
    non_outliers = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]
    
    # Apply T-test
    t_stat, p_value = ttest_ind(non_outliers[col], outliers[col], nan_policy='omit')
    
    # Print T-statistic and P-value for each column
    print(f"Column: {col}")
    print(f"T-statistic: {t_stat}, P-value: {p_value}")
    print("-" * 50)


Column: Open
T-statistic: nan, P-value: nan
--------------------------------------------------
Column: High
T-statistic: nan, P-value: nan
--------------------------------------------------
Column: Low
T-statistic: nan, P-value: nan
--------------------------------------------------
Column: Close
T-statistic: nan, P-value: nan
--------------------------------------------------
Column: Adj Close
T-statistic: nan, P-value: nan
--------------------------------------------------
Column: Volume
T-statistic: -6.710897424597064, P-value: 3.223371922604479e-11
--------------------------------------------------


In [27]:
alpha = 0.05  # Significance level

# Check if p-value is less than alpha
if p_value < alpha:
    print(f"Column: {col} - Significant difference between outliers and non-outliers")
else:
    print(f"Column: {col} - No significant difference between outliers and non-outliers")


Column: Volume - Significant difference between outliers and non-outliers


# Engineer New Features from Date

In [28]:
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['IsWeekend'] = df['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)


# Scale Numerical Features

In [32]:
numerical_cols = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])


In [33]:
df[numerical_cols]

,Open,High,Low,Close,Adj Close,Volume
0,-1.447772,-1.441465,-1.510141,-1.522047,-1.522047,1.062268
1,-1.579589,-1.452453,-1.556931,-1.416167,-1.416167,1.156404
2,-1.405553,-1.399802,-1.377121,-1.426885,-1.426885,0.599365
3,-1.400944,-1.444029,-1.510420,-1.560481,-1.560481,0.657948
4,-1.522898,-1.552262,-1.639627,-1.566302,-1.566302,1.641237
...,...,...,...,...,...,...
1004,-0.157532,0.021787,-0.131848,0.075199,0.075199,1.921872
1005,0.128133,0.303632,0.122471,0.352278,0.352278,2.115057
1006,0.269076,0.244113,0.131215,0.096818,0.096818,1.370702
1007,0.021942,0.036071,-0.075292,-0.123810,-0.123810,0.760603


In [41]:


# Save the preprocessed DataFrame as a CSV file
df.to_csv('preprocessed_data.csv', index=False)  # index=False excludes the index from being written
